In [9]:
import os
from urllib.parse import quote_plus
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
import pymysql

# -----------------------------------------------------------------------------
# 1. Database Configuration & Connection
# -----------------------------------------------------------------------------
# Force reload of environment variables
load_dotenv(override=True)

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
# Use explicit loopback IP '127.0.0.1' to avoid Windows hostname lookup bugs
DB_HOST = os.getenv('DB_HOST', '127.0.0.1').strip()
DB_PORT = os.getenv('DB_PORT', '3306').strip()
DB_NAME = os.getenv('DB_NAME')

assert DB_PASSWORD is not None, "Error: DB_PASSWORD is missing in .env file"

# URL-encode password (handles special characters like @ safely)
encoded_password = quote_plus(DB_PASSWORD)

# Create SQLAlchemy engine
connection_string = f"mysql+pymysql://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string, pool_pre_ping=True)

# Test connection
with engine.connect() as conn:
    print("✅ Successfully connected to MySQL database engine!")

# -----------------------------------------------------------------------------
# 2. Data Directory & Path Setup
# -----------------------------------------------------------------------------
DATA_DIR = os.path.join("..", "data")
print(f"Data Directory Path: {os.path.abspath(DATA_DIR)}")

# -----------------------------------------------------------------------------
# 3. Process & Clean Datasets
# -----------------------------------------------------------------------------
# A. Orders Dataset
orders_path = os.path.join(DATA_DIR, "olist_orders_dataset.csv")
orders_df = pd.read_csv(orders_path)

datetime_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    orders_df[col] = pd.to_datetime(orders_df[col])

orders_df.drop_duplicates(subset=['order_id'], inplace=True)
print(f"Orders Dataset Shape: {orders_df.shape}")

# B. Customers Dataset
customers_path = os.path.join(DATA_DIR, "olist_customers_dataset.csv")
customers_df = pd.read_csv(customers_path)
customers_df.drop_duplicates(subset=['customer_id'], inplace=True)

# C. Order Items Dataset
order_items_path = os.path.join(DATA_DIR, "olist_order_items_dataset.csv")
order_items_df = pd.read_csv(order_items_path)
order_items_df['shipping_limit_date'] = pd.to_datetime(order_items_df['shipping_limit_date'])

# D. Payments Dataset
payments_path = os.path.join(DATA_DIR, "olist_order_payments_dataset.csv")
payments_df = pd.read_csv(payments_path)

print("Data Cleaning Complete:")
print(f"• Customers: {customers_df.shape}")
print(f"• Order Items: {order_items_df.shape}")
print(f"• Payments: {payments_df.shape}")

# -----------------------------------------------------------------------------
# 4. Load Cleaned Data into MySQL
# -----------------------------------------------------------------------------
print("Pushing clean tables to MySQL...")

orders_df.to_sql('orders', con=engine, if_exists='replace', index=False)
customers_df.to_sql('customers', con=engine, if_exists='replace', index=False)
order_items_df.to_sql('order_items', con=engine, if_exists='replace', index=False)
payments_df.to_sql('payments', con=engine, if_exists='replace', index=False)

print("🚀 Step 1 Complete: All 4 base Olist tables loaded into MySQL successfully!")

✅ Successfully connected to MySQL database engine!
Data Directory Path: c:\Projects\CustomerShopping&BehaviourAnalysis\data
Orders Dataset Shape: (99441, 8)
Data Cleaning Complete:
• Customers: (99441, 5)
• Order Items: (112650, 7)
• Payments: (103886, 5)
Pushing clean tables to MySQL...
🚀 Step 1 Complete: All 4 base Olist tables loaded into MySQL successfully!
